# Gold Layer 3 — Hospital Operational Activity

### Goal

Create an analysis-ready Gold dataset that summarises hospital care-unit activity over time.

The final table will contain **one row per activity date and care unit** and will support analysis of:

- daily patient presence
- patient arrivals and departures
- transfer activity
- ICU-related activity
- high-activity periods
- weekday and weekend patterns

The dataset measures **recorded operational activity**, not true hospital capacity or staffed-bed occupancy.

### Grain

**One row = one `activity_date` + `careunit`**

The main source is the Silver `transfers` table because it records the time patients spend within individual hospital care units.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Catalog and schemas
silver_schema = "cdr_silver.processed_mimic_iv"
gold_schema = "cdr_gold.analytics_mimic_iv"

# Load required Silver tables
admissions = spark.table(f"{silver_schema}.admissions")
transfers = spark.table(f"{silver_schema}.transfers")
icustays = spark.table(f"{silver_schema}.icustays")
services = spark.table(f"{silver_schema}.services")

print("Admissions:", admissions.count())
print("Transfers:", transfers.count())
print("ICU stays:", icustays.count())
print("Services:", services.count())

## 1. Validate Transfer Records

The `transfers` table is the main source for care-unit activity.

Before building daily activity records, we check:
- transfer event types
- missing or unknown care units
- missing start/end times
- invalid time ordering

This helps separate genuine inpatient care-unit intervals from administrative or incomplete movement records.

In [0]:
# Transfer quality summary by event type
transfer_quality = (
    transfers
    .groupBy("eventtype")
    .agg(
        F.count("*").alias("rows"),
        F.sum((F.col("careunit") == "UNKNOWN").cast("int"))
            .alias("unknown_careunit"),
        F.sum(F.col("careunit").isNull().cast("int"))
            .alias("null_careunit"),
        F.sum(F.col("hadm_id").isNull().cast("int"))
            .alias("missing_hadm_id"),
        F.sum(F.col("intime").isNull().cast("int"))
            .alias("missing_intime"),
        F.sum(F.col("outtime").isNull().cast("int"))
            .alias("missing_outtime"),
        F.sum(
            (
                F.col("intime").isNotNull()
                & F.col("outtime").isNotNull()
                & (F.col("outtime") < F.col("intime"))
            ).cast("int")
        ).alias("invalid_time_order")
    )
    .orderBy("eventtype")
)

display(transfer_quality)

## 2. Build Inpatient Care-Unit Intervals

Daily operational activity is based on `admit` and `transfer` records with a known hospital admission and care unit.

Records with an unknown care unit are excluded from unit-level analysis. For the small number of admission records with a missing end time, the next recorded hospital movement is used as the inferred end of the interval.

The original transfer records remain unchanged; this logic is applied only when constructing the Gold analytical dataset.

In [0]:
# Order movement records within each hospital admission
movement_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy("intime", "transfer_id")
)

transfer_sequence = (
    transfers
    .withColumn(
        "next_event_intime",
        F.lead("intime").over(movement_window)
    )
)

# Build inpatient care-unit intervals
careunit_intervals = (
    transfer_sequence
    .filter(
        F.col("eventtype").isin("admit", "transfer")
        & F.col("hadm_id").isNotNull()
        & F.col("intime").isNotNull()
        & F.col("careunit").isNotNull()
        & (F.col("careunit") != "UNKNOWN")
    )
    .withColumn(
        "outtime_inferred",
        F.col("outtime").isNull()
    )
    .withColumn(
        "effective_outtime",
        F.coalesce(
            F.col("outtime"),
            F.col("next_event_intime")
        )
    )
    .filter(
        F.col("effective_outtime").isNotNull()
        & (F.col("effective_outtime") >= F.col("intime"))
    )
    .select(
        "subject_id",
        "hadm_id",
        "transfer_id",
        "eventtype",
        "careunit",
        "intime",
        "effective_outtime",
        "outtime_inferred"
    )
)

# Validate resulting interval set
display(
    careunit_intervals.agg(
        F.count("*").alias("valid_intervals"),
        F.countDistinct("transfer_id").alias("unique_transfer_ids"),
        F.countDistinct("hadm_id").alias("distinct_admissions"),
        F.sum(F.col("outtime_inferred").cast("int"))
            .alias("inferred_outtimes")
    )
)

## 3. Validate Care-Unit Interval Timing

The `transfers` table is treated as the primary source for care-unit activity.

Transfer timestamps do not always align exactly with the formal admission and discharge timestamps. In particular, Emergency Department Observation activity frequently begins at the recorded hospital discharge time, reflecting differences between operational movement records and admission-level timestamps.

Admission boundaries are therefore used for quality checking rather than automatically clipping transfer intervals. Only clearly inconsistent transfer durations are handled as data-quality exceptions.

In [0]:
# Use transfer intervals as the operational activity source
operational_intervals = (
    careunit_intervals
    .withColumn(
        "interval_hours",
        (
            F.col("effective_outtime").cast("long")
            - F.col("intime").cast("long")
        ) / 3600
    )
)

# Validate interval durations before creating daily activity records
display(
    operational_intervals.agg(
        F.count("*").alias("intervals"),
        F.countDistinct("transfer_id").alias("unique_transfer_ids"),
        F.countDistinct("hadm_id").alias("distinct_admissions"),
        F.sum(
            F.col("outtime_inferred").cast("int")
        ).alias("inferred_outtimes"),
        F.sum(
            (F.col("interval_hours") == 0).cast("int")
        ).alias("zero_duration_intervals"),
        F.sum(
            (F.col("interval_hours") > 24 * 100).cast("int")
        ).alias("over_100_day_intervals"),
        F.expr(
            """
            percentile_approx(
                interval_hours,
                array(0.5, 0.95, 0.99)
            )
            """
        ).alias("duration_hours_percentiles"),
        F.max("interval_hours").alias("max_interval_hours")
    )
)

## 4. Review Clearly Inconsistent Care-Unit Intervals

Most care-unit intervals have reasonable durations, including some genuine long hospital stays.

Hospital admission timestamps are used only as a quality check. We identify intervals that sit completely outside their associated hospital admission and review their care units and event types before deciding whether they should be excluded.

Long intervals are not automatically treated as invalid because some patients have legitimately long admissions.

In [0]:
# Add hospital admission boundaries for quality checking
interval_quality = (
    operational_intervals
    .join(
        admissions.select(
            "hadm_id",
            F.col("admittime").alias("hospital_admittime"),
            F.col("dischtime").alias("hospital_dischtime")
        ),
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "starts_after_discharge",
        F.col("intime") > F.col("hospital_dischtime")
    )
    .withColumn(
        "ends_before_admission",
        F.col("effective_outtime") < F.col("hospital_admittime")
    )
    .withColumn(
        "fully_outside_admission",
        F.col("starts_after_discharge")
        | F.col("ends_before_admission")
    )
)

# Summary of clearly non-overlapping intervals
display(
    interval_quality
    .filter(F.col("fully_outside_admission"))
    .groupBy("eventtype", "careunit")
    .agg(
        F.count("*").alias("intervals"),
        F.countDistinct("hadm_id").alias("admissions"),
        F.sum(
            F.col("starts_after_discharge").cast("int")
        ).alias("starts_after_discharge"),
        F.sum(
            F.col("ends_before_admission").cast("int")
        ).alias("ends_before_admission")
    )
    .orderBy(F.desc("intervals"))
)

## 5. Review Extremely Long Care-Unit Intervals

A small number of care-unit intervals exceed 100 days.

Long duration alone does not make an interval invalid because some hospital admissions may genuinely be prolonged. These records are compared with their associated hospital admission duration to identify obvious timestamp anomalies without removing legitimate long stays.

In [0]:
# Review care-unit intervals longer than 100 days
long_intervals = (
    interval_quality
    .filter(
        F.col("interval_hours") > 24 * 100
    )
    .withColumn(
        "hospital_los_hours",
        (
            F.col("hospital_dischtime").cast("long")
            - F.col("hospital_admittime").cast("long")
        ) / 3600
    )
    .withColumn(
        "start_difference_hours",
        (
            F.col("intime").cast("long")
            - F.col("hospital_admittime").cast("long")
        ) / 3600
    )
    .withColumn(
        "end_difference_hours",
        (
            F.col("effective_outtime").cast("long")
            - F.col("hospital_dischtime").cast("long")
        ) / 3600
    )
    .select(
        "subject_id",
        "hadm_id",
        "transfer_id",
        "eventtype",
        "careunit",
        "intime",
        "effective_outtime",
        "interval_hours",
        "hospital_admittime",
        "hospital_dischtime",
        "hospital_los_hours",
        "start_difference_hours",
        "end_difference_hours",
        "outtime_inferred"
    )
    .orderBy(
        F.desc("interval_hours")
    )
)

display(long_intervals)

## 6. Create Final Operational Intervals

The reviewed transfer intervals are retained unless they show a clear extreme timestamp anomaly.

Intervals longer than 100 days are excluded only when their recorded end time also extends more than 30 days beyond the associated hospital discharge. This targets the small group of clearly inconsistent records while retaining genuine long hospital stays.

Zero-duration intervals are also excluded because they do not represent measurable care-unit presence.

In [0]:
# Add a conservative flag for extreme timestamp anomalies
intervals_checked = (
    interval_quality
    .withColumn(
        "end_after_discharge_hours",
        (
            F.col("effective_outtime").cast("long")
            - F.col("hospital_dischtime").cast("long")
        ) / 3600
    )
    .withColumn(
        "extreme_timing_anomaly",
        F.when(
            F.col("hospital_dischtime").isNotNull()
            & (F.col("interval_hours") > 24 * 100)
            & (F.col("end_after_discharge_hours") > 24 * 30),
            True
        ).otherwise(False)
    )
)

# Create final interval set for operational activity
final_intervals = (
    intervals_checked
    .filter(
        (~F.col("extreme_timing_anomaly"))
        & (F.col("interval_hours") > 0)
    )
    .select(
        "subject_id",
        "hadm_id",
        "transfer_id",
        "eventtype",
        "careunit",
        "intime",
        "effective_outtime",
        "outtime_inferred",
        "interval_hours"
    )
)

# Validate final interval set
display(
    intervals_checked.agg(
        F.count("*").alias("original_intervals"),
        F.sum(
            F.col("extreme_timing_anomaly").cast("int")
        ).alias("extreme_anomalies")
    )
)

display(
    final_intervals.agg(
        F.count("*").alias("final_intervals"),
        F.countDistinct("transfer_id").alias("unique_transfer_ids"),
        F.countDistinct("hadm_id").alias("distinct_admissions"),
        F.sum(
            F.col("outtime_inferred").cast("int")
        ).alias("inferred_outtimes"),
        F.min("interval_hours").alias("min_interval_hours"),
        F.max("interval_hours").alias("max_interval_hours")
    )
)

## 7. Create Daily Care-Unit Patient Presence

Each valid care-unit interval is expanded into the calendar dates on which the patient was present in that unit.

An interval ending exactly at midnight is not counted as presence on the following day. This avoids creating an additional patient-day when the patient leaves at the start of a new calendar date.

These daily records provide the base for calculating patient load by care unit and date.

In [0]:
# Expand each interval into the dates with recorded patient presence
daily_presence = (
    final_intervals

    .withColumn(
        "presence_start_date",
        F.to_date("intime")
    )

    # Subtract one second so an interval ending exactly at midnight
    # is not counted as presence on the following day
    .withColumn(
        "presence_end_date",
        F.to_date(
            F.expr("effective_outtime - INTERVAL 1 SECOND")
        )
    )

    .withColumn(
        "activity_date",
        F.explode(
            F.sequence(
                F.col("presence_start_date"),
                F.col("presence_end_date")
            )
        )
    )

    .select(
        "subject_id",
        "hadm_id",
        "transfer_id",
        "careunit",
        "activity_date"
    )
)

# Validate expanded daily presence records
display(
    daily_presence.agg(
        F.count("*").alias("interval_day_rows"),
        F.countDistinct(
            "hadm_id",
            "careunit",
            "activity_date"
        ).alias("unique_admission_unit_days"),
        F.countDistinct("hadm_id").alias("distinct_admissions"),
        F.countDistinct("subject_id").alias("distinct_patients"),
        F.countDistinct("careunit").alias("distinct_careunits"),
        F.min("activity_date").alias("first_activity_date"),
        F.max("activity_date").alias("last_activity_date")
    )
)

## 8. Validate Repeated Daily Presence Records

A hospital admission may have multiple transfer intervals involving the same care unit on the same calendar date.

For daily patient-load measures, an admission should only contribute once to a care unit on a given date. Repeated interval-day records are therefore identified before the daily operational dataset is aggregated.

In [0]:
# Count transfer intervals contributing to each admission-unit-day
admission_unit_day_check = (
    daily_presence
    .groupBy(
        "hadm_id",
        "careunit",
        "activity_date"
    )
    .agg(
        F.count("*").alias("interval_rows"),
        F.countDistinct("transfer_id").alias("distinct_transfer_intervals")
    )
)

# Summarise repeated admission-unit-day presence
display(
    admission_unit_day_check.agg(
        F.count("*").alias("unique_admission_unit_days"),

        F.sum(
            (F.col("interval_rows") > 1).cast("int")
        ).alias("repeated_admission_unit_days"),

        F.sum(
            F.when(
                F.col("interval_rows") > 1,
                F.col("interval_rows") - 1
            ).otherwise(0)
        ).alias("excess_interval_day_rows"),

        F.max("interval_rows").alias("max_intervals_same_admission_unit_day")
    )
)

# Distribution of how many intervals contribute to one admission-unit-day
display(
    admission_unit_day_check
    .groupBy("interval_rows")
    .count()
    .orderBy("interval_rows")
)

## 9. Create Unique Daily Patient Presence

For daily patient-load measures, each hospital admission is counted once per care unit and calendar date.

Multiple transfer intervals involving the same admission and care unit on the same day are collapsed into a single presence record. The original interval-level dataset is retained separately for movement and transfer activity measures.

In [0]:
# Create one presence record per admission, care unit and date
daily_admission_presence = (
    daily_presence
    .select(
        "subject_id",
        "hadm_id",
        "careunit",
        "activity_date"
    )
    .dropDuplicates(
        ["hadm_id", "careunit", "activity_date"]
    )
)

# Validate deduplicated daily presence
display(
    daily_admission_presence.agg(
        F.count("*").alias("admission_unit_day_rows"),
        F.countDistinct(
            "hadm_id",
            "careunit",
            "activity_date"
        ).alias("unique_admission_unit_days"),
        F.countDistinct("hadm_id").alias("distinct_admissions"),
        F.countDistinct("subject_id").alias("distinct_patients"),
        F.countDistinct("careunit").alias("distinct_careunits"),
        F.min("activity_date").alias("first_activity_date"),
        F.max("activity_date").alias("last_activity_date")
    )
)

## 10. Aggregate Daily Care-Unit Patient Presence

The deduplicated admission-level presence records are aggregated to the final Gold grain of one row per `activity_date` and `careunit`.

Two related measures are retained:

- `active_patient_count` counts distinct patients recorded in the care unit on that date.
- `distinct_admission_count` counts distinct hospital admissions represented in the care unit on that date.

These measures describe recorded care-unit activity and should not be interpreted as staffed-bed occupancy or formal hospital capacity.

In [0]:
# Aggregate patient presence to one row per date and care unit
daily_presence_summary = (
    daily_admission_presence
    .groupBy(
        "activity_date",
        "careunit"
    )
    .agg(
        F.countDistinct("subject_id")
            .alias("active_patient_count"),

        F.countDistinct("hadm_id")
            .alias("distinct_admission_count")
    )
)

# Validate the daily care-unit grain
display(
    daily_presence_summary.agg(
        F.count("*").alias("careunit_day_rows"),

        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.countDistinct("careunit")
            .alias("distinct_careunits"),

        F.min("activity_date")
            .alias("first_activity_date"),

        F.max("activity_date")
            .alias("last_activity_date"),

        F.min("active_patient_count")
            .alias("min_active_patients"),

        F.max("active_patient_count")
            .alias("max_active_patients")
    )
)

## 11. Create Daily Care-Unit Movement Measures

Care-unit movement activity is derived from the start and end of each valid transfer interval.

An interval start represents an arrival into the care unit, while an interval end represents a departure from the care unit.

Arrivals are also separated by source event type:
- `admit` represents entry associated with hospital admission
- `transfer` represents movement into the unit from another hospital location

These measures describe recorded operational movements rather than formal hospital admission or discharge counts.

In [0]:
# Create daily care-unit arrival measures
daily_arrivals = (
    final_intervals
    .withColumn(
        "activity_date",
        F.to_date("intime")
    )
    .groupBy(
        "activity_date",
        "careunit"
    )
    .agg(
        F.count("*")
            .alias("patient_arrival_count"),

        F.sum(
            (F.col("eventtype") == "admit").cast("int")
        ).alias("admission_arrival_count"),

        F.sum(
            (F.col("eventtype") == "transfer").cast("int")
        ).alias("transfer_arrival_count")
    )
)

# Create daily care-unit departure measures
daily_departures = (
    final_intervals
    .withColumn(
        "activity_date",
        F.to_date("effective_outtime")
    )
    .groupBy(
        "activity_date",
        "careunit"
    )
    .agg(
        F.count("*")
            .alias("patient_departure_count")
    )
)

# Validate movement summaries
display(
    daily_arrivals.agg(
        F.sum("patient_arrival_count")
            .alias("total_arrivals"),

        F.sum("admission_arrival_count")
            .alias("admission_arrivals"),

        F.sum("transfer_arrival_count")
            .alias("transfer_arrivals"),

        F.count("*")
            .alias("arrival_careunit_days")
    )
)

display(
    daily_departures.agg(
        F.sum("patient_departure_count")
            .alias("total_departures"),

        F.count("*")
            .alias("departure_careunit_days")
    )
)

## 12. Build Daily Operational Activity Backbone

Daily patient presence, arrivals and departures are combined at the common grain of `activity_date` and `careunit`.

The backbone includes any care-unit date with recorded patient presence or movement activity. Missing measures are filled with zero so movement-only dates are retained rather than being lost during the joins.

Net patient flow and total patient movement are also calculated as operational activity measures.

In [0]:
# Create the complete set of care-unit dates across presence and movement activity
activity_keys = (
    daily_presence_summary
    .select("activity_date", "careunit")

    .union(
        daily_arrivals.select(
            "activity_date",
            "careunit"
        )
    )

    .union(
        daily_departures.select(
            "activity_date",
            "careunit"
        )
    )

    .dropDuplicates(
        ["activity_date", "careunit"]
    )
)

# Combine presence, arrivals and departures
daily_operational_activity = (
    activity_keys

    .join(
        daily_presence_summary,
        on=["activity_date", "careunit"],
        how="left"
    )

    .join(
        daily_arrivals,
        on=["activity_date", "careunit"],
        how="left"
    )

    .join(
        daily_departures,
        on=["activity_date", "careunit"],
        how="left"
    )

    # Replace missing activity measures with zero
    .fillna(
        0,
        subset=[
            "active_patient_count",
            "distinct_admission_count",
            "patient_arrival_count",
            "admission_arrival_count",
            "transfer_arrival_count",
            "patient_departure_count"
        ]
    )

    # Derived movement measures
    .withColumn(
        "net_patient_flow",
        F.col("patient_arrival_count")
        - F.col("patient_departure_count")
    )

    .withColumn(
        "total_patient_movement",
        F.col("patient_arrival_count")
        + F.col("patient_departure_count")
    )
)

# Validate the combined care-unit-day backbone
display(
    daily_operational_activity.agg(
        F.count("*").alias("careunit_day_rows"),

        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.countDistinct("careunit")
            .alias("distinct_careunits"),

        F.sum("patient_arrival_count")
            .alias("total_arrivals"),

        F.sum("patient_departure_count")
            .alias("total_departures"),

        F.sum(
            F.when(
                F.col("active_patient_count") == 0,
                1
            ).otherwise(0)
        ).alias("movement_only_careunit_days"),

        F.min("activity_date")
            .alias("first_activity_date"),

        F.max("activity_date")
            .alias("last_activity_date")
    )
)

## 13. Map ICU Stays to Operational Care-Unit Intervals

The `transfers` table remains the canonical source for care-unit location.

ICU stays are mapped to valid transfer intervals using hospital admission ID and overlapping timestamps. This assigns ICU-related activity to the operational care unit recorded in the transfer data rather than relying on potentially different care-unit labels between the two sources.

For incomplete ICU stays with no recorded end time, activity is retained within the ICU intake date. Only the overlapping portion of each ICU stay and transfer interval is used.

In [0]:
# Prepare ICU intervals for overlap matching
icu_intervals = (
    icustays

    .filter(
        F.col("stay_id").isNotNull()
        & F.col("hadm_id").isNotNull()
        & F.col("intime").isNotNull()
    )

    .select(
        "subject_id",
        "hadm_id",
        "stay_id",
        F.col("intime").alias("icu_intime"),
        F.col("outtime").alias("icu_outtime")
    )

    # Incomplete ICU stays are retained within their intake date
    .withColumn(
        "icu_effective_outtime",
        F.when(
            F.col("icu_outtime").isNull(),
            F.to_timestamp(
                F.date_add(
                    F.to_date("icu_intime"),
                    1
                )
            )
        ).otherwise(
            F.col("icu_outtime")
        )
    )
)


# Match ICU stays to operational transfer intervals
icu_transfer_overlap = (
    final_intervals.alias("t")

    .join(
        F.broadcast(icu_intervals).alias("i"),
        (
            (F.col("t.hadm_id") == F.col("i.hadm_id"))
            & (F.col("t.intime") < F.col("i.icu_effective_outtime"))
            & (F.col("t.effective_outtime") > F.col("i.icu_intime"))
        ),
        how="inner"
    )

    .withColumn(
        "overlap_start",
        F.greatest(
            F.col("t.intime"),
            F.col("i.icu_intime")
        )
    )

    .withColumn(
        "overlap_end",
        F.least(
            F.col("t.effective_outtime"),
            F.col("i.icu_effective_outtime")
        )
    )

    .filter(
        F.col("overlap_end") > F.col("overlap_start")
    )

    .select(
        F.col("i.subject_id").alias("subject_id"),
        F.col("i.hadm_id").alias("hadm_id"),
        F.col("i.stay_id").alias("stay_id"),
        F.col("t.transfer_id").alias("transfer_id"),
        F.col("t.careunit").alias("careunit"),
        "overlap_start",
        "overlap_end"
    )
)


# Expand overlapping ICU-transfer intervals into operational care-unit dates
icu_operational_daily = (
    icu_transfer_overlap

    .withColumn(
        "start_date",
        F.to_date("overlap_start")
    )

    .withColumn(
        "end_date",
        F.to_date(
            F.expr("overlap_end - INTERVAL 1 SECOND")
        )
    )

    .withColumn(
        "activity_date",
        F.explode(
            F.sequence(
                F.col("start_date"),
                F.col("end_date")
            )
        )
    )

    .select(
        "subject_id",
        "hadm_id",
        "stay_id",
        "careunit",
        "activity_date"
    )

    .dropDuplicates(
        ["stay_id", "careunit", "activity_date"]
    )
)


# Validate mapped ICU activity
display(
    icu_operational_daily.agg(
        F.count("*")
            .alias("icu_operational_unit_days"),

        F.countDistinct("stay_id")
            .alias("mapped_icu_stays"),

        F.countDistinct(
            "stay_id",
            "activity_date"
        ).alias("mapped_icu_stay_days"),

        F.countDistinct("careunit")
            .alias("mapped_careunits"),

        F.min("activity_date")
            .alias("first_activity_date"),

        F.max("activity_date")
            .alias("last_activity_date")
    )
)


# Identify ICU stays not mapped to a valid operational interval
unmapped_icu_stays = (
    icustays
    .select("stay_id")
    .distinct()

    .join(
        icu_operational_daily
        .select("stay_id")
        .distinct(),
        on="stay_id",
        how="left_anti"
    )
)

display(
    unmapped_icu_stays.agg(
        F.count("*").alias("unmapped_icu_stays")
    )
)

## 14. Validate Unmapped ICU Activity

The ICU-to-transfer overlap mapping captures all but one ICU stay.

The remaining unmapped stay is checked against the transfer quality rules to confirm that it belongs to an admission containing an already identified extreme timestamp anomaly. This ensures the ICU activity is excluded consistently with the operational transfer cleaning.

In [0]:
# Admissions containing an extreme transfer timing anomaly
extreme_anomaly_admissions = (
    intervals_checked
    .filter(
        F.col("extreme_timing_anomaly")
    )
    .select("hadm_id")
    .distinct()
    .withColumn(
        "linked_extreme_transfer_anomaly",
        F.lit(1)
    )
)


# Review ICU stays that could not be mapped
unmapped_icu_validation = (
    icustays.alias("i")

    .join(
        unmapped_icu_stays.alias("u"),
        on="stay_id",
        how="inner"
    )

    .join(
        admissions.select(
            "hadm_id",
            F.col("admittime").alias("hospital_admittime"),
            F.col("dischtime").alias("hospital_dischtime")
        ),
        on="hadm_id",
        how="left"
    )

    .join(
        extreme_anomaly_admissions,
        on="hadm_id",
        how="left"
    )

    .fillna(
        0,
        subset=["linked_extreme_transfer_anomaly"]
    )

    .withColumn(
        "icu_los_hours",
        (
            F.col("i.outtime").cast("long")
            - F.col("i.intime").cast("long")
        ) / 3600
    )

    .withColumn(
        "hospital_los_hours",
        (
            F.col("hospital_dischtime").cast("long")
            - F.col("hospital_admittime").cast("long")
        ) / 3600
    )
)


# Confirm unmapped ICU activity is linked to the known anomaly
display(
    unmapped_icu_validation.agg(
        F.countDistinct("stay_id")
            .alias("unmapped_icu_stays"),

        F.sum("linked_extreme_transfer_anomaly")
            .alias("linked_to_extreme_transfer_anomaly")
    )
)


# Show the affected stay for auditability
display(
    unmapped_icu_validation.select(
        "stay_id",
        F.col("i.subject_id").alias("subject_id"),
        F.col("i.hadm_id").alias("hadm_id"),
        F.col("i.first_careunit").alias("icu_first_careunit"),
        F.col("i.intime").alias("icu_intime"),
        F.col("i.outtime").alias("icu_outtime"),
        "icu_los_hours",
        "hospital_admittime",
        "hospital_dischtime",
        "hospital_los_hours",
        "linked_extreme_transfer_anomaly"
    )
)

## 15. Add ICU-Related Activity to the Operational Backbone

Mapped ICU activity is aggregated to the operational `activity_date + careunit` grain and joined to the existing transfer-derived backbone.

The transfer dataset remains the source of care-unit location, while ICU stays provide additional ICU-specific activity measures.

Care-unit dates without mapped ICU activity are recorded with zero ICU counts.

In [0]:
# Aggregate mapped ICU activity to the operational care-unit-day grain
daily_icu_operational_activity = (
    icu_operational_daily
    .groupBy(
        "activity_date",
        "careunit"
    )
    .agg(
        F.countDistinct("stay_id")
            .alias("active_icu_stay_count"),

        F.countDistinct("subject_id")
            .alias("active_icu_patient_count")
    )
)


# Add ICU-related activity to the transfer-derived operational backbone
operational_with_icu = (
    daily_operational_activity

    .join(
        daily_icu_operational_activity,
        on=["activity_date", "careunit"],
        how="left"
    )

    .fillna(
        0,
        subset=[
            "active_icu_stay_count",
            "active_icu_patient_count"
        ]
    )

    .withColumn(
        "icu_activity_flag",
        F.when(
            F.col("active_icu_stay_count") > 0,
            1
        ).otherwise(0)
    )
)


# Validate final operational + ICU grain
display(
    operational_with_icu.agg(
        F.count("*").alias("careunit_day_rows"),

        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.countDistinct("careunit")
            .alias("distinct_careunits"),

        F.sum("icu_activity_flag")
            .alias("careunit_days_with_icu_activity"),

        F.sum("active_icu_stay_count")
            .alias("mapped_icu_unit_day_activity"),

        F.min("activity_date")
            .alias("first_activity_date"),

        F.max("activity_date")
            .alias("last_activity_date")
    )
)

## 16. Add Operational and Calendar Features

Additional analytical fields are created for reporting and trend analysis.

`daily_patient_turnover` represents total recorded arrivals and departures relative to the number of active patients in the care unit on that date. It is left null when no active patient presence is recorded.

Calendar fields support analysis of yearly, monthly, weekly and weekday/weekend activity patterns.

In [0]:
# Add operational and calendar features
gold_activity_base = (
    operational_with_icu

    .withColumn(
        "daily_patient_turnover",
        F.when(
            F.col("active_patient_count") > 0,
            F.col("total_patient_movement")
            / F.col("active_patient_count")
        ).otherwise(F.lit(None).cast("double"))
    )

    .withColumn(
        "activity_year",
        F.year("activity_date")
    )

    .withColumn(
        "activity_month",
        F.month("activity_date")
    )

    .withColumn(
        "activity_week",
        F.weekofyear("activity_date")
    )

    .withColumn(
        "day_of_week",
        F.date_format("activity_date", "EEEE")
    )

    .withColumn(
        "weekend_flag",
        F.when(
            F.dayofweek("activity_date").isin(1, 7),
            1
        ).otherwise(0)
    )

    .withColumn(
        "security_careunit",
        F.col("careunit")
    )
)

# Validate derived fields
display(
    gold_activity_base.agg(
        F.count("*")
            .alias("careunit_day_rows"),

        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.sum(
            F.col("daily_patient_turnover").isNull().cast("int")
        ).alias("null_turnover_days"),

        F.min("daily_patient_turnover")
            .alias("min_patient_turnover"),

        F.max("daily_patient_turnover")
            .alias("max_patient_turnover"),

        F.sum("weekend_flag")
            .alias("weekend_careunit_days")
    )
)

## 17. Review Operational Activity Distributions

Daily operational measures are reviewed before defining a high-activity indicator.

Because care units vary substantially in size and activity, the high-activity threshold is based on the observed distribution of the dataset rather than a clinical or capacity-based cutoff.

The main measures reviewed are:
- active patient count
- total patient movement
- patient arrivals
- patient departures
- daily patient turnover

In [0]:
# Review distributions of core operational activity measures
display(
    gold_activity_base.agg(
        F.expr(
            """
            percentile_approx(
                active_patient_count,
                array(0.5, 0.75, 0.9, 0.95, 0.99)
            )
            """
        ).alias("active_patient_percentiles"),

        F.expr(
            """
            percentile_approx(
                total_patient_movement,
                array(0.5, 0.75, 0.9, 0.95, 0.99)
            )
            """
        ).alias("movement_percentiles"),

        F.expr(
            """
            percentile_approx(
                patient_arrival_count,
                array(0.5, 0.75, 0.9, 0.95, 0.99)
            )
            """
        ).alias("arrival_percentiles"),

        F.expr(
            """
            percentile_approx(
                patient_departure_count,
                array(0.5, 0.75, 0.9, 0.95, 0.99)
            )
            """
        ).alias("departure_percentiles"),

        F.expr(
            """
            percentile_approx(
                daily_patient_turnover,
                array(0.5, 0.75, 0.9, 0.95, 0.99)
            )
            """
        ).alias("turnover_percentiles")
    )
)

## 18. Create Care-Unit Relative High-Activity Indicators

High operational activity is defined relative to the historical activity distribution of each care unit.

For each care unit, the 95th percentile is calculated separately for:
- active patient count
- total patient movement

A care-unit day is flagged as high activity when either measure is **strictly above** its care-unit-specific 95th percentile.

Using a strict comparison avoids over-classifying days when many observations share the same discrete count at the percentile threshold.

These flags are dataset-relative operational indicators and should not be interpreted as hospital capacity thresholds or clinical risk measures.

In [0]:
# Calculate care-unit-specific 95th percentile activity thresholds
careunit_activity_thresholds = (
    gold_activity_base
    .groupBy("careunit")
    .agg(
        F.expr(
            "percentile_approx(active_patient_count, 0.95)"
        ).alias("p95_active_patient_count"),

        F.expr(
            "percentile_approx(total_patient_movement, 0.95)"
        ).alias("p95_total_patient_movement")
    )
)

# Add final care-unit-relative high-activity indicators
gold_activity_flagged = (
    gold_activity_base

    .join(
        careunit_activity_thresholds,
        on="careunit",
        how="left"
    )

    .withColumn(
        "high_patient_load_flag",
        F.when(
            F.col("active_patient_count")
            > F.col("p95_active_patient_count"),
            1
        ).otherwise(0)
    )

    .withColumn(
        "high_movement_flag",
        F.when(
            F.col("total_patient_movement")
            > F.col("p95_total_patient_movement"),
            1
        ).otherwise(0)
    )

    .withColumn(
        "high_activity_flag",
        F.when(
            (F.col("high_patient_load_flag") == 1)
            | (F.col("high_movement_flag") == 1),
            1
        ).otherwise(0)
    )
)

# Validate final high-activity indicators
display(
    gold_activity_flagged.agg(
        F.count("*").alias("careunit_day_rows"),

        F.sum("high_patient_load_flag")
            .alias("high_patient_load_days"),

        F.sum("high_movement_flag")
            .alias("high_movement_days"),

        F.sum("high_activity_flag")
            .alias("high_activity_days"),

        (
            F.sum("high_activity_flag") / F.count("*")
        ).alias("high_activity_share")
    )
)

# Review thresholds by care unit
display(
    careunit_activity_thresholds
    .orderBy("careunit")
)

## 19. Assemble the Final Gold Dataset

The validated operational measures are combined into the final Gold dataset.

The published table contains one row per `activity_date` and `careunit`, with measures for:
- recorded patient presence
- arrivals and departures
- transfer activity
- ICU-related activity
- patient turnover
- care-unit-relative high-activity indicators
- calendar fields for trend analysis

The dataset represents recorded operational activity and should not be interpreted as staffed-bed capacity, formal occupancy, or clinical risk.

In [0]:
# Assemble final Gold 3 dataset
hospital_operational_activity_gold = (
    gold_activity_flagged

    .select(
        # Grain
        "activity_date",
        "careunit",

        # Patient presence
        "active_patient_count",
        "distinct_admission_count",

        # Movement activity
        "patient_arrival_count",
        "admission_arrival_count",
        "transfer_arrival_count",
        "patient_departure_count",
        "net_patient_flow",
        "total_patient_movement",
        "daily_patient_turnover",

        # ICU-related activity
        "active_icu_stay_count",
        "active_icu_patient_count",
        "icu_activity_flag",

        # Relative activity thresholds and flags
        "p95_active_patient_count",
        "p95_total_patient_movement",
        "high_patient_load_flag",
        "high_movement_flag",
        "high_activity_flag",

        # Calendar fields
        "activity_year",
        "activity_month",
        "activity_week",
        "day_of_week",
        "weekend_flag",

        # Security field
        "security_careunit"
    )

    .withColumn(
        "_gold_timestamp",
        F.current_timestamp()
    )
)

# Final structural validation
display(
    hospital_operational_activity_gold.agg(
        F.count("*").alias("rows"),

        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.countDistinct("careunit")
            .alias("distinct_careunits"),

        F.sum(
            F.col("activity_date").isNull().cast("int")
        ).alias("null_activity_dates"),

        F.sum(
            F.col("careunit").isNull().cast("int")
        ).alias("null_careunits"),

        F.sum(
            F.col("daily_patient_turnover").isNull().cast("int")
        ).alias("null_turnover_days"),

        F.sum("high_activity_flag")
            .alias("high_activity_days"),

        F.sum("icu_activity_flag")
            .alias("icu_activity_days")
    )
)

display(
    hospital_operational_activity_gold
    .orderBy(
        "activity_date",
        "careunit"
    )
    .limit(20)
)

## 20. Final Logical Validation

Before publishing the Gold table, the derived operational measures are checked for internal consistency.

This confirms that:
- arrival components reconcile to total arrivals
- movement and net-flow measures reconcile
- ICU patient counts do not exceed recorded active patient counts
- activity flags agree with their underlying measures
- security care-unit values match the analytical care-unit key
- no negative activity counts have been introduced

In [0]:
# Final logical consistency checks
final_validation = (
    hospital_operational_activity_gold
    .agg(
        # Arrival components must reconcile
        F.sum(
            (
                F.col("patient_arrival_count")
                != (
                    F.col("admission_arrival_count")
                    + F.col("transfer_arrival_count")
                )
            ).cast("int")
        ).alias("arrival_component_mismatches"),

        # Total movement must equal arrivals + departures
        F.sum(
            (
                F.col("total_patient_movement")
                != (
                    F.col("patient_arrival_count")
                    + F.col("patient_departure_count")
                )
            ).cast("int")
        ).alias("movement_mismatches"),

        # Net flow must equal arrivals - departures
        F.sum(
            (
                F.col("net_patient_flow")
                != (
                    F.col("patient_arrival_count")
                    - F.col("patient_departure_count")
                )
            ).cast("int")
        ).alias("net_flow_mismatches"),

        # ICU patients should not exceed all active patients
        F.sum(
            (
                F.col("active_icu_patient_count")
                > F.col("active_patient_count")
            ).cast("int")
        ).alias("icu_patient_count_mismatches"),

        # ICU flag must agree with ICU stay count
        F.sum(
            (
                F.col("icu_activity_flag")
                != F.when(
                    F.col("active_icu_stay_count") > 0,
                    1
                ).otherwise(0)
            ).cast("int")
        ).alias("icu_flag_mismatches"),

        # High-activity flag must agree with component flags
        F.sum(
            (
                F.col("high_activity_flag")
                != F.when(
                    (F.col("high_patient_load_flag") == 1)
                    | (F.col("high_movement_flag") == 1),
                    1
                ).otherwise(0)
            ).cast("int")
        ).alias("high_activity_flag_mismatches"),

        # Security field must match the care-unit key
        F.sum(
            (
                F.col("security_careunit")
                != F.col("careunit")
            ).cast("int")
        ).alias("security_careunit_mismatches"),

        # Check for negative count measures
        F.sum(
            (
                (F.col("active_patient_count") < 0)
                | (F.col("distinct_admission_count") < 0)
                | (F.col("patient_arrival_count") < 0)
                | (F.col("patient_departure_count") < 0)
                | (F.col("total_patient_movement") < 0)
                | (F.col("active_icu_stay_count") < 0)
                | (F.col("active_icu_patient_count") < 0)
            ).cast("int")
        ).alias("negative_count_rows"),

        # Thresholds should be available for every published row
        F.sum(
            (
                F.col("p95_active_patient_count").isNull()
                | F.col("p95_total_patient_movement").isNull()
            ).cast("int")
        ).alias("missing_activity_thresholds")
    )
)

display(final_validation)

## 21. Publish Gold Table

The validated operational activity dataset is written to Unity Catalog as a managed Delta table.

The published table uses the final Gold grain of one row per `activity_date` and `careunit`.

In [0]:
# Final Gold table name
gold_table = f"{gold_schema}.hospital_operational_activity"

# Write final dataset to Unity Catalog
(
    hospital_operational_activity_gold
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print(f"Published: {gold_table}")

## 22. Add Unity Catalog Metadata

Table- and column-level descriptions are added to the published Gold table to document its analytical grain, measure definitions, security field and important interpretation limits.

The metadata makes the dataset easier to understand in Catalog Explorer, SQL, dashboards and Genie without requiring users to inspect the transformation notebook.

In [0]:
# Add table-level description
table_comment = """
Gold analytical table summarising recorded hospital operational activity at one row per activity_date and careunit.

The table is derived primarily from MIMIC-IV transfer intervals, with ICU activity mapped through overlapping ICU stays. It supports analysis of daily patient presence, patient movement, ICU-related activity, turnover, calendar trends and unusually high operational activity.

Measures represent recorded operational activity and should not be interpreted as staffed-bed capacity, formal occupancy rates or clinical risk.
""".strip()

spark.sql(
    f"""
    COMMENT ON TABLE {gold_table}
    IS '{table_comment.replace("'", "''")}'
    """
)


# Column-level metadata
column_comments = {
    "activity_date":
        "Calendar date of recorded care-unit operational activity.",

    "careunit":
        "Operational hospital care unit recorded in the transfers source. Together with activity_date, this defines the table grain.",

    "active_patient_count":
        "Distinct patients with recorded presence in the care unit on the activity date. This is an operational presence measure and not a staffed-bed occupancy count.",

    "distinct_admission_count":
        "Distinct hospital admissions represented in the care unit on the activity date.",

    "patient_arrival_count":
        "Number of valid care-unit intervals beginning in the care unit on the activity date.",

    "admission_arrival_count":
        "Number of care-unit arrivals on the activity date originating from transfer records with event type admit.",

    "transfer_arrival_count":
        "Number of care-unit arrivals on the activity date originating from transfer records with event type transfer.",

    "patient_departure_count":
        "Number of valid care-unit intervals ending in the care unit on the activity date.",

    "net_patient_flow":
        "Patient arrivals minus patient departures for the care unit and activity date.",

    "total_patient_movement":
        "Total recorded patient movement, calculated as patient arrivals plus patient departures.",

    "daily_patient_turnover":
        "Total patient movement divided by active patient count for the care unit and date. Null when no active patient presence is recorded.",

    "active_icu_stay_count":
        "Distinct ICU stays overlapping transfer-defined operational activity in the care unit on the activity date.",

    "active_icu_patient_count":
        "Distinct patients with ICU stays overlapping transfer-defined operational activity in the care unit on the activity date.",

    "icu_activity_flag":
        "Indicator equal to 1 when at least one mapped ICU stay is active in the care unit on the activity date; otherwise 0.",

    "p95_active_patient_count":
        "Historical 95th-percentile active patient count calculated separately for the care unit.",

    "p95_total_patient_movement":
        "Historical 95th-percentile total patient movement calculated separately for the care unit.",

    "high_patient_load_flag":
        "Dataset-relative indicator equal to 1 when active patient count is strictly above the care-unit-specific historical 95th percentile.",

    "high_movement_flag":
        "Dataset-relative indicator equal to 1 when total patient movement is strictly above the care-unit-specific historical 95th percentile.",

    "high_activity_flag":
        "Dataset-relative operational indicator equal to 1 when either patient load or patient movement is strictly above its care-unit-specific 95th-percentile threshold. This is not a capacity or clinical risk measure.",

    "activity_year":
        "Year extracted from activity_date. MIMIC dates are deidentified and shifted.",

    "activity_month":
        "Calendar month number extracted from activity_date.",

    "activity_week":
        "Week-of-year number extracted from activity_date.",

    "day_of_week":
        "Day-of-week name derived from activity_date.",

    "weekend_flag":
        "Indicator equal to 1 for Saturday or Sunday and 0 for weekdays.",

    "security_careunit":
        "Care-unit security field intended for care-unit-based row-level access control.",

    "_gold_timestamp":
        "Timestamp recording when the Gold dataset was generated."
}


# Apply column comments
for column_name, comment in column_comments.items():
    escaped_comment = comment.replace("'", "''")

    spark.sql(
        f"""
        ALTER TABLE {gold_table}
        ALTER COLUMN `{column_name}`
        COMMENT '{escaped_comment}'
        """
    )


print(
    f"Unity Catalog metadata added for "
    f"{len(column_comments)} columns in {gold_table}"
)

## 23. Final Persisted Table Validation

The published Unity Catalog table is reloaded and validated independently of the in-memory transformation DataFrame.

This confirms that:
- the final `activity_date + careunit` grain is preserved
- key operational totals remain unchanged after persistence
- the expected Gold columns are present
- table- and column-level metadata are available in Unity Catalog

In [0]:
# Reload the persisted Gold table
hospital_operational_activity_persisted = (
    spark.table(gold_table)
)

# Validate persisted analytical results
display(
    hospital_operational_activity_persisted.agg(
        F.count("*")
            .alias("rows"),

        F.countDistinct(
            "activity_date",
            "careunit"
        ).alias("unique_careunit_days"),

        F.countDistinct("careunit")
            .alias("distinct_careunits"),

        F.sum("patient_arrival_count")
            .alias("total_arrivals"),

        F.sum("patient_departure_count")
            .alias("total_departures"),

        F.sum("high_activity_flag")
            .alias("high_activity_days"),

        F.sum("icu_activity_flag")
            .alias("icu_activity_days"),

        F.sum(
            F.col("daily_patient_turnover").isNull().cast("int")
        ).alias("null_turnover_days"),

        F.min("activity_date")
            .alias("first_activity_date"),

        F.max("activity_date")
            .alias("last_activity_date")
    )
)


# Check table-level Unity Catalog description
display(
    spark.sql(
        f"DESCRIBE DETAIL {gold_table}"
    ).select(
        "name",
        "description"
    )
)


# Check column-level comments
column_metadata = (
    spark.sql(
        f"DESCRIBE TABLE {gold_table}"
    )
    .filter(
        F.col("col_name").isin(
            hospital_operational_activity_persisted.columns
        )
    )
)

display(column_metadata)


# Validate metadata coverage
display(
    column_metadata.agg(
        F.count("*").alias("columns"),

        F.sum(
            F.col("comment").isNotNull().cast("int")
        ).alias("columns_with_comments"),

        F.sum(
            F.col("comment").isNull().cast("int")
        ).alias("columns_without_comments")
    )
)